# Car logbook generator

Builds an ATO-style motor vehicle logbook spreadsheet from what you earned.

**How to use it:** run each cell in order with the play button on the left. Fill
in the boxes as you go. The last cell downloads your spreadsheet.

Nothing you type leaves this page. The notebook runs in your own account, the
file is built there, and it downloads straight to your device.

Before you rely on it, check the rows against your platform statements: the ATO
expects a logbook to record journeys you actually made.


In [ ]:
#@title 1. Set up (run this first, takes about 20 seconds) { display-mode: "form" }
import os, subprocess, sys

REPO = "https://github.com/Mr-Yash-01/car-logbook"   #@param {type:"string"}
HERE = "/content/car-logbook" if os.path.isdir("/content") else "car-logbook"

if os.path.isdir(os.path.join(HERE, "logbook")):
    pass                                   # already here from an earlier run
elif os.path.isdir("logbook"):
    HERE = "."                             # Binder and local Jupyter start in the repo
else:
    got = subprocess.run(["git", "clone", "--depth", "1", "-q", REPO + ".git", HERE],
                         capture_output=True, text=True)
    if got.returncode:
        raise SystemExit(
            f"Could not fetch the code from {REPO}.\n\n"
            "Check that the address in the REPO box above is right, and that the "
            "repository is public - a private one cannot be read from here.\n\n"
            f"git said: {got.stderr.strip().splitlines()[-1] if got.stderr.strip() else 'nothing'}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl"], check=True)
sys.path.insert(0, os.path.abspath(HERE))
from logbook.notebook import make_logbook
print("Ready.")


## 2. Your car

The ATO asks for make, model, engine size and plate, so the logbook clearly
belongs to one particular car. Engine size is on your registration papers.


In [ ]:
#@title Car and owner { display-mode: "form" }
make = "Toyota"  #@param {type:"string"}
model = "Corolla"  #@param {type:"string"}
engine_capacity = "1.8L"  #@param {type:"string"}
registration = "ABC123"  #@param {type:"string"}
owner_name = ""  #@param {type:"string"}
year_of_manufacture = ""  #@param {type:"string"}
print(f"{make} {model}, {engine_capacity}, plate {registration or '(none yet)'}")


## 3. The period and the odometer

Pick the first day and how many weeks. The ATO wants at least 12 continuous
weeks the first time you keep a logbook, and that stretch should look like your
normal driving.

The two odometer readings are the numbers on your dashboard on the first day and
the last. The gap between them is every kilometre the car travelled, business
and private together.


In [ ]:
#@title Period and odometer { display-mode: "form" }
start_date = "01/07/2025"  #@param {type:"string"}
weeks = 12  #@param {type:"integer"}
start_odometer = 45230  #@param {type:"integer"}
end_odometer = 55230  #@param {type:"integer"}
print(f"{end_odometer - start_odometer:,} km over {weeks} weeks")


## 4. What you earned

Your driving income for a typical month, before expenses. Business kilometres
are worked out from this, so use the figure from your platform statement. A part
month at either end is scaled down automatically.


In [ ]:
#@title Income { display-mode: "form" }
income_per_month = 5000  #@param {type:"number"}
print(f"${income_per_month:,.0f} a month")


## 5. How you drive

**Work days** matter most: business travel is recorded on those days and no
others. A day you do not name here carries private travel only, so name every
day you normally drive for income.

**The share** is how much of a work day's driving is for income. The rest of
that day counts as private — the shopping on the way home, the detour to pick
someone up. Most drivers sit between 70% and 95%.

**The daily limits** keep every day believable: nothing in the logbook goes
above them.

If your answers cannot hold the kilometres, you will be told what does not fit
and the closest workable pattern is used instead — one that adds work days or
lifts a limit, never one that records fewer business kilometres.


In [ ]:
#@title Driving pattern { display-mode: "form" }
work_type = "rideshare"  #@param ["rideshare", "delivery", "both"]
work_days = "Fri,Sat,Sun"  #@param {type:"string"}
business_share_on_work_days = "85%"  #@param {type:"string"}
max_km_per_day = 420  #@param {type:"integer"}
max_business_km_per_day = 350  #@param {type:"integer"}
max_private_km_per_day = 150  #@param {type:"integer"}
city = "brisbane"  #@param ["brisbane", "sydney", "melbourne", "perth", "adelaide", "canberra", "hobart", "darwin"]
print(f"{work_type} on {work_days}, {business_share_on_work_days} of a work day is business")


## 6. Make it

The shuffle number varies the daily distances so the logbook does not read as a
pattern. The same number always gives you the same logbook; change it to
reshuffle the days.

If your period crosses 30 June it covers two financial years. Leaving the box
ticked gives each year its own sheet, which is easier at tax time.


In [ ]:
#@title Build and download { display-mode: "form" }
sheet_per_financial_year = True  #@param {type:"boolean"}
seed = 42  #@param {type:"integer"}

make_logbook(
    start_date=start_date, weeks=weeks,
    income_per_month=income_per_month,
    start_odometer=start_odometer, end_odometer=end_odometer,
    make=make, model=model, engine_capacity=engine_capacity,
    registration=registration, owner_name=owner_name,
    year_of_manufacture=year_of_manufacture,
    work_type=work_type, work_days=work_days,
    business_share_on_work_days=business_share_on_work_days,
    max_km_per_day=max_km_per_day,
    max_business_km_per_day=max_business_km_per_day,
    max_private_km_per_day=max_private_km_per_day,
    city=city, sheet_per_financial_year=sheet_per_financial_year, seed=seed)


## Optional: check it against the ATO's requirements

Run this to see the finished file measured against the record-keeping rules on
the ATO's logbook method page.


In [ ]:
#@title Audit the file { display-mode: "form" }
import glob, os, subprocess, sys
newest = max(glob.glob("*.xlsx"), key=os.path.getmtime)
print(subprocess.run([sys.executable, os.path.join(HERE, "audit_ato.py"), newest],
                     capture_output=True, text=True).stdout)
